# Structured Output

Structured Output allows us to instruct an LLM to generate responses in a predefined format or schema instead of returning plain, unstructured text.

This makes the model's output predictable, consistent, and easier to parse and use in subsequent applications.

LangChain supports multiple schema types and methods for enforcing structured output.

## Pydantic

Pydantic is a Python library used for data validation and defining structured data models.

Pydantic models provide a rich feature set, including:

- **Field Validation:** Ensures that the generated data follows the expected data types and constraints.
- **Descriptions:** Allows us to describe what each field represents.
- **Nested Structures:** Supports complex data structures containing other models.

**Key Idea:** Instead of receiving a free-form text response, we can ask the LLM to return data following a predefined Pydantic schema.

In [2]:
import os
from langchain.chat_models import init_chat_model

os.environ["GROQ_API_KEY"]=os.getenv("GROQ_API_KEY")
model = init_chat_model("groq:openai/gpt-oss-20b")

In [5]:
from pydantic import BaseModel, Field

class Movie(BaseModel):
    title: str = Field(description="The title of the movie")
    year: int = Field(description="This year the movie was released")
    hero:str= Field("who is the hero of the movie")
    heroine:str = Field("who is the heroine of the movie")
    director: str = Field(description="The director of the movie")
    rating: float = Field(description="The movies rating out of 10")

modelwithstructuredop=model.with_structured_output(Movie)
response = modelwithstructuredop.invoke("tell me about the kannada movie gaja")
response

Movie(title='Gaja', year=2021, hero='Diganth', heroine='Ramya', director='Karthik H. Gaja', rating=6.7)

In [6]:
from pydantic import BaseModel, Field

class Movie(BaseModel):
    title: str = Field(...,description="The title of the movie")
    year: int = Field(...,description="This year the movie was released")
    hero:str= Field(...,description="who is the hero of the movie")
    heroine:str = Field(...,description="who is the heroine of the movie")
    director: str = Field(...,description="The director of the movie")
    rating: float = Field(...,description="The movies rating out of 10")

modelwithstructuredop=model.with_structured_output(Movie,include_raw=True)
response = modelwithstructuredop.invoke("tell me about the kannada movie gaja")
response

{'raw': AIMessage(content='', additional_kwargs={'reasoning_content': 'User wants info about Kannada movie "Gaja". Likely the film "Gaja" (2020?) There\'s a Kannada movie "Gaja" starring Raghavendra Rajkumar? Wait, there\'s "Gaja" 2020 starring Raghavendra Rajkumar? Let\'s recall. There\'s a 2021 Kannada film "Gaja" starring Raghavendra Rajkumar? Actually, there\'s a 2020 Kannada film "Gaja" starring Raghavendra Rajkumar and Nethra Raghunath? Let\'s search memory. There is a Kannada film "Gaja" (2020) starring Raghavendra Rajkumar, with music by Arjun Janya? Maybe not.\n\nAlternatively, there is a 2023 film "Gaja" starring Raghavendra Rajkumar? Wait, there\'s a film "Gaja" 2020 directed by B. S. Harsha? Hmm.\n\nLet\'s recall: "Gaja" (2020) is a Kannada action drama directed by R. Shankar? Not sure.\n\nLet\'s think: The user might want a summary: plot, cast, crew, release year, rating. We have a function to provide movie data. We can call the function with director, hero, heroine, ratin

### nested structures


In [7]:
from pydantic import BaseModel, Field


class Actor(BaseModel):
    name: str
    role: str


class MovieDetails(BaseModel):
    title: str
    year: int
    cast: list[Actor]
    genres: list[str]
    budget: float | None = Field(
        None,
        description="Budget in millions USD"
    )


# Bind the Pydantic schema to the LLM
model_with_structure = model.with_structured_output(MovieDetails)


# Invoke the model with a query
response = model_with_structure.invoke(
    "Provide details about the movie Inception"
)

# Display the structured response
response

MovieDetails(title='Inception', year=2010, cast=[Actor(name='Leonardo DiCaprio', role='Dom Cobb'), Actor(name='Joseph Gordon-Levitt', role='Arthur'), Actor(name='Elliot Page', role='Ariadne'), Actor(name='Tom Hardy', role='Eames'), Actor(name='Ken Watanabe', role='Saito'), Actor(name='Cillian Murphy', role='Robert Fischer'), Actor(name='Marion Cotillard', role='Mal')], genres=['Action', 'Adventure', 'Sci-Fi', 'Thriller'], budget=160000000.0)

# TypedDict

`TypedDict` is a Python feature used to define the expected structure of a dictionary, including its keys and their expected data types.

In LangChain, TypedDict can be used to define schemas for structured LLM outputs.

Unlike Pydantic, TypedDict primarily provides type information and does not perform runtime data validation by itself.

## Why use TypedDict?

- **Define Structure:** Specifies which keys a dictionary should contain and their expected data types.
- **Lightweight:** Provides a simpler alternative to Pydantic with less code.
- **Type Hints:** Helps IDEs and static type checkers understand the expected dictionary structure.
- **Structured Output:** Allows an LLM to generate responses following a predefined schema.
- **Descriptions:** Using `Annotated`, we can provide descriptions of individual fields.

## When to use TypedDict?

Use TypedDict when:

- You need a simple schema for structured LLM outputs.
- You don't require Pydantic's runtime validation features.
- You want to define the expected keys and data types of a dictionary.
- You prefer a lightweight approach with minimal code complexity.

## TypedDict vs Pydantic

| Feature | TypedDict | Pydantic |
|---|---|---|
| Define data structure | Yes | Yes |
| Type hints | Yes | Yes |
| Runtime validation | No, not by itself | Yes |
| Data parsing | No automatic parsing | Yes |
| Field descriptions | Using `Annotated` | Using `Field` |
| Complexity | Lightweight | More feature-rich |

**Key Takeaway:** Use TypedDict when you need a simple dictionary schema for structured output. Use Pydantic when you need runtime validation, data parsing, and more advanced validation rules.

In [8]:
from typing_extensions import TypedDict, Annotated


class MovieDict(TypedDict):
    """A movie with details."""

    title: Annotated[str, ..., "The title of the movie"]
    year: Annotated[int, ..., "The year the movie was released"]
    director: Annotated[str, ..., "The director of the movie"]
    rating: Annotated[float, ..., "The movie's rating out of 10"]


# Bind the TypedDict schema to the LLM
model_withtypedict = model.with_structured_output(MovieDict)

# Invoke the model with a query
response = model_withtypedict.invoke(
    "Please provide the details of the movie Avengers"
)

# Display the response
response

{'director': 'Joss Whedon', 'rating': 8, 'title': 'The Avengers', 'year': 2008}

In [9]:
from typing_extensions import TypedDict, Annotated


# Define the structure of an actor
class Actor(TypedDict):
    name: str
    role: str


# Define the structure of movie information
class MovieDetails(TypedDict):
    title: str
    year: int

    # A list containing Actor dictionaries
    cast: list[Actor]

    # A list of movie genres
    genres: list[str]

    # Budget can be a float or None
    budget: Annotated[
        float | None,
        ...,
        "Budget in millions USD"
    ]


# Bind the TypedDict schema to the LLM
model_with_structure = model.with_structured_output(MovieDetails)


# Ask the LLM to provide information about Inception
response = model_with_structure.invoke(
    "Provide details about the movie Inception"
)


# Display the structured response
print(response)

{'budget': 160, 'cast': [{'name': 'Leonardo DiCaprio', 'role': 'Dom Cobb'}, {'name': 'Joseph Gordon-Levitt', 'role': 'Arthur'}, {'name': 'Elliot Page', 'role': 'Ariadne'}, {'name': 'Tom Hardy', 'role': 'Eames'}, {'name': 'Ken Watanabe', 'role': 'Saito'}, {'name': 'Cillian Murphy', 'role': 'Robert Fischer'}], 'genres': ['Action', 'Adventure', 'Sci-Fi'], 'title': 'Inception', 'year': 2010}


In [14]:
# =====================================================
# STEP 1: IMPORT REQUIRED LIBRARIES
# =====================================================

import os
from dotenv import load_dotenv

from pydantic import BaseModel, Field
from langchain.agents import create_agent


# =====================================================
# STEP 2: LOAD API KEY
# =====================================================

# Load environment variables from .env
load_dotenv()

# Make sure GROQ_API_KEY is available
assert os.getenv("GROQ_API_KEY"), "GROQ_API_KEY is missing"


# =====================================================
# STEP 3: DEFINE PYDANTIC DATA MODEL
# =====================================================

class ContactInfo(BaseModel):

    # Expected name of the person
    name: str = Field(description="Name of the person")

    # Expected email address
    email: str = Field(description="Email address of the person")

    # Expected phone number
    phone: str = Field(description="Phone number of the person")


# =====================================================
# STEP 4: CREATE THE AI AGENT
# =====================================================

agent = create_agent(

    # Specify the LLM provider and model
    model="groq:openai/gpt-oss-20b",

    # Tell the agent to return responses following ContactInfo
    response_format=ContactInfo
)


# =====================================================
# STEP 5: PROVIDE INPUT TO THE AGENT
# =====================================================

result = agent.invoke({

    "messages": [
        {
            "role": "user",
            "content": "Extract the information from this: John Doe, johndoe@gmail.com, 2509829340"
        }
    ]

})


# =====================================================
# STEP 6: EXTRACT STRUCTURED RESPONSE
# =====================================================

# Retrieve the structured Pydantic object
print(result)

contact = result["structured_response"]

print(contact)



{'messages': [HumanMessage(content='Extract the information from this: John Doe, johndoe@gmail.com, 2509829340', additional_kwargs={}, response_metadata={}, id='3a8a31ad-0313-4ab8-8635-5d8ae451b6fc'), AIMessage(content='{"name":"John Doe","email":"johndoe@gmail.com","phone":"2509829340"}', additional_kwargs={'reasoning_content': 'We need to output JSON object with fields name, email, phone. Must be compact JSON. Must include all required fields. So output: {"name":"John Doe","email":"johndoe@gmail.com","phone":"2509829340"}. Ensure JSON valid. Use compact formatting.'}, response_metadata={'token_usage': {'completion_tokens': 95, 'prompt_tokens': 227, 'total_tokens': 322, 'completion_time': 0.103548458, 'completion_tokens_details': {'reasoning_tokens': 60}, 'prompt_time': 0.015409443, 'prompt_tokens_details': None, 'queue_time': 0.285299718, 'total_time': 0.118957901}, 'model_name': 'openai/gpt-oss-20b', 'system_fingerprint': 'fp_ef00694abe', 'service_tier': 'on_demand', 'finish_reason'